# Week 5: Evaluations & Testing

**Lab companion to [week_05.md](../agenda/week_05.md)**

In this lab, you will:
1. Evaluate retrieval quality (precision, recall)
2. Evaluate generation quality (LLM-as-judge)
3. Build automated test suites
4. Compare different configurations

In [2]:
from openai import OpenAI
from dotenv import load_dotenv
import json

load_dotenv()
client = OpenAI()

print("✓ Ready!")

✓ Ready!


## 1. Creating a Test Dataset

In [3]:
# Create a mock RAG system for testing
knowledge_base = {
    "doc1": "Python is a programming language created by Guido van Rossum in 1991. It focuses on code readability.",
    "doc2": "JavaScript was created by Brendan Eich in 1995. It runs in web browsers.",
    "doc3": "Machine learning uses algorithms to learn patterns from data without explicit programming.",
    "doc4": "Deep learning uses neural networks with many layers to learn complex patterns.",
    "doc5": "REST APIs use HTTP methods like GET, POST, PUT, DELETE for communication."
}

# Golden test dataset: (question, expected_relevant_docs, expected_answer_contains)
test_cases = [
    {
        "question": "Who created Python?",
        "relevant_docs": ["doc1"],
        "expected_facts": ["Guido van Rossum", "1991"]
    },
    {
        "question": "What is deep learning?",
        "relevant_docs": ["doc3", "doc4"],
        "expected_facts": ["neural networks", "layers"]
    },
    {
        "question": "What HTTP methods do REST APIs use?",
        "relevant_docs": ["doc5"],
        "expected_facts": ["GET", "POST"]
    }
]

print(f"Test dataset: {len(test_cases)} cases")

Test dataset: 3 cases


## 2. Retrieval Evaluation

Measure how well retrieval finds the right documents.

In [4]:
import numpy as np

def get_embedding(text: str) -> list:
    response = client.embeddings.create(
        model="text-embedding-3-small",
        input=text
    )
    return response.data[0].embedding

def cosine_similarity(a: list, b: list) -> float:
    a, b = np.array(a), np.array(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

# Pre-compute embeddings
doc_embeddings = {doc_id: get_embedding(text) for doc_id, text in knowledge_base.items()}

def retrieve(query: str, top_k: int = 2) -> list:
    """Simple retrieval for testing."""
    query_emb = get_embedding(query)

    scores = [
        (doc_id, cosine_similarity(query_emb, doc_emb))
        for doc_id, doc_emb in doc_embeddings.items()
    ]
    scores.sort(key=lambda x: x[1], reverse=True)

    return [doc_id for doc_id, _ in scores[:top_k]]

# Test
results = retrieve("Who invented Python?")
print(f"Retrieved: {results}")

Retrieved: ['doc1', 'doc2']


In [5]:
def precision_at_k(retrieved: list, relevant: list) -> float:
    """What fraction of retrieved docs are relevant?"""
    if not retrieved:
        return 0.0
    relevant_set = set(relevant)
    return len([d for d in retrieved if d in relevant_set]) / len(retrieved)

def recall_at_k(retrieved: list, relevant: list) -> float:
    """What fraction of relevant docs were retrieved?"""
    if not relevant:
        return 0.0
    retrieved_set = set(retrieved)
    return len([d for d in relevant if d in retrieved_set]) / len(relevant)

def mrr(retrieved: list, relevant: list) -> float:
    """Mean Reciprocal Rank - how early is the first relevant doc?"""
    relevant_set = set(relevant)
    for i, doc in enumerate(retrieved, 1):
        if doc in relevant_set:
            return 1.0 / i
    return 0.0

# Evaluate retrieval
print("Retrieval Evaluation:")
print("=" * 50)

all_precision = []
all_recall = []
all_mrr = []

for case in test_cases:
    retrieved = retrieve(case["question"], top_k=2)
    relevant = case["relevant_docs"]

    p = precision_at_k(retrieved, relevant)
    r = recall_at_k(retrieved, relevant)
    m = mrr(retrieved, relevant)

    all_precision.append(p)
    all_recall.append(r)
    all_mrr.append(m)

    print(f"Q: {case['question'][:40]}...")
    print(f"   Retrieved: {retrieved} | Expected: {relevant}")
    print(f"   Precision: {p:.2f} | Recall: {r:.2f} | MRR: {m:.2f}")
    print()

print("=" * 50)
print(f"Average Precision: {np.mean(all_precision):.2f}")
print(f"Average Recall: {np.mean(all_recall):.2f}")
print(f"Average MRR: {np.mean(all_mrr):.2f}")

Retrieval Evaluation:
Q: Who created Python?...
   Retrieved: ['doc1', 'doc2'] | Expected: ['doc1']
   Precision: 0.50 | Recall: 1.00 | MRR: 1.00

Q: What is deep learning?...
   Retrieved: ['doc4', 'doc3'] | Expected: ['doc3', 'doc4']
   Precision: 1.00 | Recall: 1.00 | MRR: 1.00

Q: What HTTP methods do REST APIs use?...
   Retrieved: ['doc5', 'doc3'] | Expected: ['doc5']
   Precision: 0.50 | Recall: 1.00 | MRR: 1.00

Average Precision: 0.67
Average Recall: 1.00
Average MRR: 1.00


## 3. Generation Evaluation (LLM-as-Judge)

In [6]:
def generate_answer(question: str, context: str) -> str:
    """Generate answer using RAG."""
    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[
            {"role": "system", "content": "Answer based only on the provided context."},
            {"role": "user", "content": f"Context: {context}\n\nQuestion: {question}"}
        ]
    )
    return response.choices[0].message.content

def evaluate_answer_llm(question: str, answer: str, expected_facts: list) -> dict:
    """Use LLM to evaluate answer quality."""

    prompt = f"""Evaluate this Q&A pair:

Question: {question}
Answer: {answer}
Expected facts that should be mentioned: {expected_facts}

Rate the answer on:
1. Factual accuracy (1-5): Are the stated facts correct?
2. Completeness (1-5): Does it mention all expected facts?
3. Relevance (1-5): Does it answer the question asked?

Respond in JSON format:
{{
    "accuracy": <1-5>,
    "completeness": <1-5>,
    "relevance": <1-5>,
    "explanation": "<brief explanation>"
}}"""

    response = client.chat.completions.create(
        model="gpt-5-mini",
        messages=[{"role": "user", "content": prompt}],
        response_format={"type": "json_object"}
    )

    return json.loads(response.choices[0].message.content)

# Test evaluation
test_q = "Who created Python?"
context = knowledge_base["doc1"]
answer = generate_answer(test_q, context)

print(f"Q: {test_q}")
print(f"A: {answer}")
print()

eval_result = evaluate_answer_llm(test_q, answer, ["Guido van Rossum", "1991"])
print("Evaluation:")
print(json.dumps(eval_result, indent=2))

Q: Who created Python?
A: Guido van Rossum.

Evaluation:
{
  "accuracy": 5,
  "completeness": 3,
  "relevance": 5,
  "explanation": "The answer correctly names the creator, Guido van Rossum, so it is factually accurate and directly answers the question. It is incomplete, however, because it does not mention the year 1991 as expected."
}


In [7]:
# Full evaluation on test set
print("Full RAG Evaluation:")
print("=" * 60)

all_scores = []

for case in test_cases:
    question = case["question"]

    # Retrieve
    retrieved = retrieve(question, top_k=2)
    context = " ".join([knowledge_base[d] for d in retrieved])

    # Generate
    answer = generate_answer(question, context)

    # Evaluate
    eval_result = evaluate_answer_llm(question, answer, case["expected_facts"])

    avg_score = (eval_result["accuracy"] + eval_result["completeness"] + eval_result["relevance"]) / 3
    all_scores.append(avg_score)

    print(f"Q: {question}")
    print(f"A: {answer[:100]}...")
    print(f"Scores: acc={eval_result['accuracy']} comp={eval_result['completeness']} rel={eval_result['relevance']}")
    print(f"Explanation: {eval_result['explanation']}")
    print("-" * 60)

print(f"\nOverall Average Score: {np.mean(all_scores):.2f}/5")

Full RAG Evaluation:
Q: Who created Python?
A: Guido van Rossum....
Scores: acc=5 comp=2 rel=5
Explanation: The answer correctly names Guido van Rossum as Python's creator (accuracy). However it omits the expected year—Python was first released/created in 1991—so it is incomplete. It directly answers the question, so it is fully relevant.
------------------------------------------------------------
Q: What is deep learning?
A: Deep learning is an approach that uses neural networks with many layers to learn complex patterns. I...
Scores: acc=5 comp=5 rel=5
Explanation: The answer correctly states that deep learning uses neural networks with many layers and that it is a form of machine learning that learns patterns from data, so it includes the expected facts and directly answers the question.
------------------------------------------------------------
Q: What HTTP methods do REST APIs use?
A: REST APIs use HTTP methods like GET, POST, PUT, and DELETE for communication....
Scores: acc=5

## 4. A/B Testing Different Configs

In [8]:
def run_evaluation(config: dict, test_cases: list) -> dict:
    """Run full evaluation with a specific config."""
    results = {
        "retrieval_precision": [],
        "retrieval_recall": [],
        "answer_scores": []
    }

    for case in test_cases:
        # Retrieve
        retrieved = retrieve(case["question"], top_k=config["top_k"])

        # Retrieval metrics
        results["retrieval_precision"].append(precision_at_k(retrieved, case["relevant_docs"]))
        results["retrieval_recall"].append(recall_at_k(retrieved, case["relevant_docs"]))

        # Generate
        context = " ".join([knowledge_base[d] for d in retrieved])
        answer = generate_answer(case["question"], context)

        # Evaluate
        eval_r = evaluate_answer_llm(case["question"], answer, case["expected_facts"])
        avg = (eval_r["accuracy"] + eval_r["completeness"] + eval_r["relevance"]) / 3
        results["answer_scores"].append(avg)

    return {
        "config": config,
        "avg_precision": np.mean(results["retrieval_precision"]),
        "avg_recall": np.mean(results["retrieval_recall"]),
        "avg_answer_score": np.mean(results["answer_scores"])
    }

# Test different configs
configs = [
    {"top_k": 1, "name": "top_k=1"},
    {"top_k": 2, "name": "top_k=2"},
    {"top_k": 3, "name": "top_k=3"}
]

print("A/B Test Results:")
print("=" * 60)

for config in configs:
    result = run_evaluation(config, test_cases)
    print(f"Config: {config['name']}")
    print(f"  Precision: {result['avg_precision']:.2f}")
    print(f"  Recall: {result['avg_recall']:.2f}")
    print(f"  Answer Quality: {result['avg_answer_score']:.2f}/5")
    print()

A/B Test Results:
Config: top_k=1
  Precision: 1.00
  Recall: 0.83
  Answer Quality: 4.78/5

Config: top_k=2
  Precision: 0.67
  Recall: 1.00
  Answer Quality: 4.67/5

Config: top_k=3
  Precision: 0.44
  Recall: 1.00
  Answer Quality: 4.78/5



## 5. Creating an Evaluation Framework

In [9]:
class RAGEvaluator:
    """Framework for evaluating RAG systems."""

    def __init__(self, rag_fn, test_cases: list):
        """
        Args:
            rag_fn: Function that takes question and returns {answer, sources}
            test_cases: List of {question, relevant_docs, expected_facts}
        """
        self.rag_fn = rag_fn
        self.test_cases = test_cases
        self.client = OpenAI()

    def evaluate(self) -> dict:
        """Run full evaluation."""
        results = []

        for case in self.test_cases:
            rag_result = self.rag_fn(case["question"])

            # Retrieval metrics
            retrieved_docs = rag_result.get("sources", [])
            precision = precision_at_k(retrieved_docs, case["relevant_docs"])
            recall = recall_at_k(retrieved_docs, case["relevant_docs"])

            # Generation metrics
            answer_eval = self._evaluate_answer(
                case["question"],
                rag_result["answer"],
                case["expected_facts"]
            )

            results.append({
                "question": case["question"],
                "precision": precision,
                "recall": recall,
                **answer_eval
            })

        return self._aggregate(results)

    def _evaluate_answer(self, question: str, answer: str, expected: list) -> dict:
        prompt = f"""Rate this answer. Question: {question}
Answer: {answer}
Expected: {expected}
Reply JSON: {{"accuracy": 1-5, "completeness": 1-5, "relevance": 1-5}}"""

        response = self.client.chat.completions.create(
            model="gpt-5-mini",
            messages=[{"role": "user", "content": prompt}],
            response_format={"type": "json_object"}
        )
        return json.loads(response.choices[0].message.content)

    def _aggregate(self, results: list) -> dict:
        return {
            "num_cases": len(results),
            "avg_precision": np.mean([r["precision"] for r in results]),
            "avg_recall": np.mean([r["recall"] for r in results]),
            "avg_accuracy": np.mean([r["accuracy"] for r in results]),
            "avg_completeness": np.mean([r["completeness"] for r in results]),
            "avg_relevance": np.mean([r["relevance"] for r in results]),
            "details": results
        }

# Create a simple RAG function
def simple_rag(question: str) -> dict:
    retrieved = retrieve(question, top_k=2)
    context = " ".join([knowledge_base[d] for d in retrieved])
    answer = generate_answer(question, context)
    return {"answer": answer, "sources": retrieved}

# Evaluate
evaluator = RAGEvaluator(simple_rag, test_cases)
results = evaluator.evaluate()

print("Evaluation Summary:")
print(f"  Cases: {results['num_cases']}")
print(f"  Retrieval Precision: {results['avg_precision']:.2f}")
print(f"  Retrieval Recall: {results['avg_recall']:.2f}")
print(f"  Answer Accuracy: {results['avg_accuracy']:.1f}/5")
print(f"  Answer Completeness: {results['avg_completeness']:.1f}/5")
print(f"  Answer Relevance: {results['avg_relevance']:.1f}/5")

Evaluation Summary:
  Cases: 3
  Retrieval Precision: 0.67
  Retrieval Recall: 1.00
  Answer Accuracy: 5.0/5
  Answer Completeness: 4.3/5
  Answer Relevance: 5.0/5


## Summary

You learned:
- ✅ Retrieval metrics: Precision, Recall, MRR
- ✅ LLM-as-judge for generation quality
- ✅ A/B testing configurations
- ✅ Building evaluation frameworks

**Next:** [Week 6 - Security & Guardrails](week_06_security.ipynb)